# 7.3 A Protein Signature

**Machine learning series, class 3 of 4.** In 7.1 and 7.2, a handful of enrollment measures (age, BMI, general health) predicted new type 2 diabetes about as well as any model could. Blood proteins are a different kind of data: **thousands** of measurements per person, for a few thousand people. Can a panel of proteins see diabetes coming **before** the routine measures do?

This is the situation where **LASSO** earns its keep: far more predictors than cases.

### By the end of this notebook you'll be able to

1. Set **time zero** for a blood sample, and keep the outcome strictly after it
2. Explain why ordinary logistic regression **fails** with more predictors than cases, and how LASSO copes
3. Use **nested cross-validation** when there are too few cases for a separate test set
4. Ask the right question: does a new kind of data **add** anything beyond what we already know?
5. Put a **confidence interval** on a difference in AUC with the **bootstrap**
6. Check whether the proteins LASSO picks are **stable**

### 🔎 Analysis at a glance

| | |
|---|---|
| ❓ **Question** | Do blood proteins predict new type 2 diabetes within 2 years, beyond age, sex, and BMI? |
| 🧪 **The analysis** | People with Olink proteomics and 7.1's enrollment data; time zero = the later of enrollment and the blood draw; three models (age, sex, and BMI; proteins only; both), compared by nested cross-validation; bootstrap confidence intervals; the protein panel and its stability |
| 🎯 **Why** | A new kind of data is only useful for prediction if it adds to what's cheap and already known |
| ✅ **What we conclude** | How much proteins add (with its uncertainty), and which proteins carry the signal |
| ⚠️ **What it can't tell us** | Cause; or whether the panel works in other people: we don't know how people were chosen for proteomics |
| 🔗 **Connections** | **7.1** (the data, LASSO, the one-SE rule), **4.6** (reading the proteomics file, QC), **7.2** (Brier score) |

### Class plan (95 minutes)

| Time | Part | You do |
|---|---|---|
| 0:00 | **1. Key concepts**: more predictors than cases; nested cross-validation | listen, discuss |
| 0:15 | **2. The people and time zero** | run the cells |
| 0:25 | **3. The proteins** (start it, then discuss while it loads) | run, discuss |
| 0:35 | **4. Who's in** | run the cells |
| 0:40 | **5. Nested cross-validation** (start it, then discuss while it runs) | run, discuss |
| 1:00 | **6. Do proteins add anything?** | run the cells, discuss |
| 1:10 | **7. The protein panel** | run the cells, discuss |
| 1:25 | **Wrap-up** | ✋ **Your turn** |

### ⏱️ Time and 💰 cost

| | |
|---|---|
| ⏱️ **Run time** | about 20 to 40 minutes: about 5 to read the proteins, 3 to reshape them, and 5 to 20 for nested cross-validation |
| 💰 **Query cost** | the cohort query only (cents to under \$1). Age, sex, and BMI come from the file **7.1** saved; the proteins are files, not BigQuery |
| 💾 **Memory** | about 6 GB at the peak: fine on the default app (26 GB). Shut down other notebooks' kernels first |
| 📋 **Before class** | run **7.1** to the end in the same workspace (it saves `ml_t2d_2yr.csv`) |

> ⚠️ **Keep All of Us data in the Workbench.** Share only summaries, with no counts of 1 to 20. Clear outputs before committing.

## Part 1. Key concepts

### More predictors than cases

In 7.1, about 8,500 people developed diabetes, and we had 8 predictors: over 1,000 cases per predictor. Here we'll have about **5,000 proteins** and only a **few hundred** new cases: **less than one case per protein**.

Ordinary logistic regression can't cope. With more predictors than cases, it can always find a combination of proteins that separates the cases **perfectly** in the training data: it memorizes the people, and fails on anyone new. That's **overfitting** at its most extreme.

**LASSO** (7.1) adds a penalty that pulls every protein's coefficient towards zero, and sets most of them to **exactly** zero. The proteins left over are the **panel**. Cross-validation chooses how strong the penalty is.

### Nested cross-validation: when a test set is too small

7.1 kept a quarter of the people aside as a test set: about 2,000 new cases, plenty. Here a quarter would hold only **about 50** new cases: an AUC from 50 cases could be off by 0.05 or more, as large as the differences we're looking for.

**Nested cross-validation** uses every person for testing, once:

```
Outer loop: split everyone into 5 parts (folds)
  For each fold k = 1 to 5:
    training = the other 4 folds          test = fold k
    Inner loop: cross-validation on the training folds only, to choose the penalty
    Fit LASSO with that penalty on the training folds
    Predict the risk for the people in fold k     <- they were never seen
Every person now has a prediction from a model that didn't see them
```

Two loops, because two things need data the model hasn't seen: **choosing the penalty** (inner), and **measuring how good the model is** (outer). Use the same data for both and the result is too optimistic.

### Does it add anything?

A protein panel is expensive; age, sex, and BMI are free. So the question isn't "do proteins predict diabetes?" but "do they predict it **better than** what we already have?" We compare three models, on the **same people**:

| Model | Predictors | How it's fitted |
|---|---|---|
| **Clinical** | age, sex at birth, BMI | ordinary logistic regression |
| **Proteins** | every protein | LASSO |
| **Clinical + proteins** | age, sex, BMI, and every protein | LASSO, but age, sex, and BMI are **never** penalized: they're always in, and proteins must earn their place on top |

The **bootstrap** then tells us how sure we can be: resample the people 1,000 times, recompute the AUCs each time, and see how much the difference moves.

## Part 2. The people and time zero

### Setup

`glmnet` is the LASSO package (7.1 used it through tidymodels; here we use it directly, which is simpler for a matrix of thousands of proteins). `data.table` reads and reshapes the large protein file quickly.

In [ ]:
source("../aou_setup.R")  # settings, packages, and helper functions

install_if_missing(c("glmnet", "data.table"))   # first time only
suppressPackageStartupMessages(library(glmnet))

HORIZON_YEARS <- 2                       # the same 2-year window as 7.1 and 7.2
end_of_data   <- as.Date("2025-01-01")   # the CDR v9 data cutoff
CORES         <- parallel::detectCores() # processors in this app: the cross-validation uses them all
CORES

**🔍 Read the output:** the number of processors (4 on the default app).

### The cohort, and 7.1's enrollment data

The cohort query gives each person's first type 2 diabetes date and how long their health records go on (as in 7.1). Age, sex at birth, and BMI at enrollment come from the file 7.1 saved: **no new query**. Everyone in it had no type 2 diabetes at enrollment.

In [ ]:
cohort <- run_query(str_glue(read_file("../sql/t2d_cohort.sql"), AGE_DATE = "2025-01-01")) %>%
  mutate(person_id = as.character(person_id),
         t2d_date  = as.Date(first_t2d_date),
         last_ehr  = as.Date(last_ehr_date),
         death     = as.Date(death_date))

enrollment <- load_data(str_glue("ml_t2d_{HORIZON_YEARS}yr.csv"), col_types = cols(person_id = "c")) %>%
  select(person_id, enroll_date, age, sex_at_birth, bmi)

enrollment %>% summarise(people = n(), missing_age_sex_bmi = sum(!complete.cases(age, sex_at_birth, bmi)))

**🔍 Read the output:** the people in 7.1's file (152,586 in our test run), and how many are missing age, sex, or BMI: **0**, since 7.1 kept only people with them.

### Who has proteomics, and when was their blood drawn?

The proteomics **manifest** lists everyone with an Olink sample. It has **no date**, so we use the date the person's **genome** blood sample was collected (`biosample_collection_date`), from the genome QC file. Our data check found that for people with proteomics, this date is usually the enrollment day (median 0 days; 80% within 30 days).

> **An assumption, stated:** we assume the plasma for proteomics came from the same blood draw as the genome. The documents we have don't say. Most draws are at enrollment, so it rarely matters.

**Time zero** is the **later** of enrollment and the blood draw: the first moment we'd have **both** the enrollment data and the proteins. People with no draw date keep enrollment as time zero.

In [ ]:
PROT_DIR <- file.path(aou_data_dir("v9"), "multiomics", "proteomics")
QC_FILE  <- file.path(aou_data_dir("v9"), "wgs", "short_read", "snpindel", "aux", "qc", "genomics_metrics.tsv")

manifest <- read_tsv(file.path(PROT_DIR, "manifest.tsv"), col_types = cols(.default = "c"))

# Each person's earliest genome blood-draw date (written like 2019-05-01 10:30:00)
draws <- read_tsv(QC_FILE, col_types = cols(.default = "c"),
                  col_select = c(research_id, biosample_collection_date)) %>%
  transmute(person_id = research_id,
            draw_date = as.Date(parse_date_time(biosample_collection_date,
                                                orders = c("ymd HMS", "ymd HM", "ymd", "mdy HMS", "mdy HM", "mdy"),
                                                quiet = TRUE))) %>%
  filter(!is.na(draw_date)) %>%
  group_by(person_id) %>%
  summarise(draw_date = min(draw_date), .groups = "drop")

people <- enrollment %>%
  left_join(cohort %>% select(person_id, cohort, t2d_date, last_ehr, death), by = "person_id") %>%
  left_join(draws, by = "person_id") %>%
  mutate(has_proteomics = person_id %in% manifest$research_id,
         time_zero      = pmax(enroll_date, draw_date, na.rm = TRUE),     # the later of the two
         horizon_end    = time_zero + round(365.25 * HORIZON_YEARS),
         followup_end   = pmin(last_ehr, death, end_of_data, na.rm = TRUE),
         t2d_by_zero    = cohort == "case" & t2d_date <= time_zero,
         new_t2d        = cohort == "case" & t2d_date > time_zero & t2d_date <= horizon_end)

people %>%
  filter(has_proteomics) %>%
  summarise(pct_with_draw_date      = round(100 * mean(!is.na(draw_date)), 1),
            pct_time_zero_is_enroll = round(100 * mean(time_zero == enroll_date), 1))

**🔍 Read the output:** for people with proteomics in 7.1's file, the share with a readable blood-draw date, and the share whose time zero is simply their enrollment day (the draw was the same day, or earlier, or missing). Percentages only: no counts yet. We count people in Part 4, once the protein file has told us who has usable values.

**Why recompute the outcome?** 7.1 counted diagnoses in the 2 years after **enrollment**. Here the clock starts at **time zero**, which can be later. Someone diagnosed between enrollment and their blood draw already had diabetes when the proteins were measured: the proteins could reflect the diabetes, not predict it. `t2d_by_zero` finds them; they're left out.

## Part 3. The proteins

The file is the same as in 4.6: `Olink_10k_aou_v9_reps_removed.tsv.gz`, 2.6 GB compressed, in **long** format (one row per person and protein). As in 4.6, `awk` keeps only **our** people's rows while the file is decompressed, so R never holds the whole file. We keep 6 of its 36 columns.

The candidates are the people who could be in the model: proteomics, no diabetes by time zero, and followed for 2 years (or diagnosed sooner).

⏱️ About 5 minutes. Start it, then read on.

In [ ]:
candidates <- people %>%
  filter(has_proteomics, !t2d_by_zero, new_t2d | followup_end >= horizon_end)

NPX_FILE <- file.path(PROT_DIR, "npx", "batch_normalized", "Olink_10k_aou_v9_reps_removed.tsv.gz")
npx_columns <- str_split(system(str_glue("gzip -dc {shQuote(NPX_FILE)} | head -1"), intern = TRUE), "\t")[[1]]

ids_file <- file.path(tempdir(), "ids.txt")
write_lines(candidates$person_id, ids_file)

wanted   <- c("ResearchID", "OlinkID", "Assay", "NPX", "SampleQC", "AssayQC")
col_nums <- match(wanted, npx_columns)
stopifnot(!anyNA(col_nums))   # stops if a column name has changed

awk_command <- str_glue(
  "gzip -dc {shQuote(NPX_FILE)} | ",
  "awk -F'\\t' -v OFS='\\t' -v id={col_nums[1]} ",
  "'NR==FNR {{keep[$1]; next}} FNR==1 || ($id in keep) {{print {str_c('$', col_nums, collapse = ', ')}}}' ",
  "{shQuote(ids_file)} -"
)

start <- Sys.time()
npx_long <- data.table::fread(cmd = awk_command, sep = "\t", header = TRUE,
                              colClasses = c(ResearchID = "character", NPX = "numeric"))
Sys.time() - start

nrow(npx_long)                     # rows: people x proteins
data.table::uniqueN(npx_long$OlinkID)   # proteins

**🔍 Read the output:** how long it took, the number of rows (millions: one per person and protein), and the number of proteins (about 5,400).

### QC, then one row per person

As in 4.6: a value from a sample or assay that **failed** QC becomes missing. Then the long table becomes a **matrix** with one row per person and one column per protein, which is what `glmnet` needs. `data.table::dcast()` does what `pivot_wider()` does, much faster for 30 million rows; `mean` averages the rare person with two samples. ⏱️ 2 to 3 minutes.

In [ ]:
invisible(npx_long[SampleQC == "FAIL" | AssayQC == "FAIL", NPX := NA])   # data.table: set NPX to missing in these rows
protein_names <- unique(npx_long[, .(OlinkID, Assay)])        # each protein's name, for Part 7

npx_wide <- data.table::dcast(npx_long, ResearchID ~ OlinkID, value.var = "NPX",
                              fun.aggregate = function(x) mean(x, na.rm = TRUE))
rm(npx_long); invisible(gc())      # free the memory: we don't need the long table any more

proteins <- as.matrix(npx_wide[, -1])
rownames(proteins) <- npx_wide$ResearchID
proteins[is.nan(proteins)] <- NA   # the mean of only missing values is NaN: make it NA
rm(npx_wide); invisible(gc())

# Keep proteins measured in at least 90% of people (as in 4.6)
missing_share <- colMeans(is.na(proteins))
proteins <- proteins[, missing_share <= 0.10]

c(proteins_in_file = length(missing_share), proteins_kept = ncol(proteins),
  pct_values_missing = round(100 * mean(is.na(proteins)), 2))

**🔍 Read the output:** proteins in the file, how many are measured in at least 90% of people, and the share of values still missing (small). The remaining gaps are filled in Part 5, using the **training** people only.

## Part 4. Who's in

Each rule, and how many people are left after it. A step that removes 1 to 20 people is hidden, together with the row before it: otherwise the number could be worked out by subtraction.

In [ ]:
people <- people %>% mutate(has_protein_values = person_id %in% rownames(proteins))

rules <- list(
  rep(TRUE, nrow(people)),
  people$has_proteomics,
  !people$t2d_by_zero,
  people$new_t2d | people$followup_end >= people$horizon_end,
  people$has_protein_values)
names(rules) <- c("in 7.1's file (enrollment data, no type 2 diabetes at enrollment)",
                  "with proteomics",
                  "no type 2 diabetes by time zero (the later of enrollment and the blood draw)",
                  str_glue("health records for {HORIZON_YEARS} years after time zero, or diagnosed sooner"),
                  "with protein values in the file")
keep <- accumulate(rules, `&`)

flow  <- tibble(rule = names(rules), people = map_int(keep, sum)) %>% mutate(removed = lag(people) - people)
small <- which(flow$removed %in% 1:20)
flow$removed[c(small, small - 1)] <- NA     # hide the small step, and the step before it
flow$people[small - 1]            <- NA     # and the count it could be worked out from
flow

**🔍 Read the output:** the steps from 7.1's people to the people in the model. Expect a large drop at "with proteomics" (only a few thousand have it), then smaller ones. Hidden cells (`NA`) are small steps.

In [ ]:
model_people <- people[keep[[length(keep)]], ] %>%
  mutate(sex_male = as.integer(sex_at_birth == "Male"),
         y        = as.integer(new_t2d))           # 1 = diagnosed within 2 years of time zero

proteins <- proteins[model_people$person_id, ]     # the same people, in the same order
stopifnot(identical(rownames(proteins), model_people$person_id))

model_people %>%
  summarise(people = n(), new_t2d = sum(y), pct_new_t2d = round(100 * mean(y), 2),
            cases_per_protein = round(sum(y) / ncol(proteins), 2))

**🔍 Read the output:** the people in the model, how many developed type 2 diabetes within 2 years of time zero, and the **cases per protein**: well under 1. This is the "more predictors than cases" problem from Part 1.

**Note:** these people are **not** a random sample of *All of Us*: we don't know how people were chosen for proteomics. Their rate of new diabetes may differ from 7.1's (5.56%).

## Part 5. Nested cross-validation

### The folds

Each person goes in one of 5 **folds**. We split cases and non-cases separately, so every fold has the same share of new cases (like `strata =` in 7.1). We also fix the **inner** folds now (the folds inside each training set), so that the results are the same every time the notebook runs, even when the folds run in parallel.

In [ ]:
set.seed(2026)
assign_folds <- function(y, k) {
  fold <- integer(length(y))
  fold[y == 1] <- sample(rep(1:k, length.out = sum(y == 1)))   # cases, spread evenly
  fold[y == 0] <- sample(rep(1:k, length.out = sum(y == 0)))   # non-cases, spread evenly
  fold
}

y           <- model_people$y
outer_fold  <- assign_folds(y, 5)
inner_folds <- map(1:5, ~ assign_folds(y[outer_fold != .x], 5))   # each outer fold's inner folds, fixed now

tibble(fold = outer_fold, y) %>%
  group_by(fold) %>%
  summarise(people = n(), n = sum(y)) %>%
  hide_small_counts() %>%
  rename(new_t2d = n)

**🔍 Read the output:** five folds of about the same size, each with about a fifth of the new cases.

### Preparing the data inside each fold

Two steps use **averages**, so they must use the **training** people only (otherwise the test people leak into the training):

1. **Standardize** every protein (and age and BMI): subtract the training mean and divide by the training SD. LASSO's penalty is only fair if every predictor is on the same scale. It also makes the coefficients **per standard deviation**, so they can be compared.
2. **Fill gaps** with 0, which after standardizing is the protein's training **average**.

In [ ]:
standardize <- function(train, test) {
  centre <- colMeans(train, na.rm = TRUE)
  spread <- apply(train, 2, sd, na.rm = TRUE)
  spread[is.na(spread) | spread == 0] <- 1               # a protein with no spread: leave it as is
  scale_it <- function(x) {
    x <- sweep(sweep(x, 2, centre), 2, spread, "/")      # (value - mean) / SD, column by column
    x[is.na(x)] <- 0                                     # a gap becomes the training average
    x
  }
  list(train = scale_it(train), test = scale_it(test))
}

clinical <- as.matrix(model_people[, c("age", "sex_male", "bmi")])
rownames(clinical) <- model_people$person_id

### One fold

`run_fold(k)` does everything for one outer fold: prepare the data, fit the three models on the other four folds, and predict the risk for the people in fold `k`.

- `cv.glmnet()` is the **inner loop**: it tries 100 penalties by cross-validation on the training folds (`foldid` gives the inner folds), and remembers the best one, `lambda.min`.
- `penalty.factor` sets how much each column is penalized: **0** for age, sex, and BMI (always in), **1** for every protein.
- We use **`lambda.min`**, the penalty with the best cross-validated score. 7.1 used the one-SE rule (`lambda.1se`), the simplest model close to the best. With 8,000 cases that cost nothing; with a few hundred, the one-SE margin is wide and can remove real signal. You'll try it in **Your turn**.

In [ ]:
run_fold <- function(k) {
  train <- outer_fold != k
  test  <- outer_fold == k
  inner <- inner_folds[[k]]                              # the inner loop's folds

  prot <- standardize(proteins[train, ], proteins[test, ])
  clin <- standardize(clinical[train, ], clinical[test, ])

  # 1. Clinical: ordinary logistic regression
  clin_fit  <- glm(y[train] ~ ., data = as.data.frame(clin$train), family = binomial)
  clin_risk <- predict(clin_fit, newdata = as.data.frame(clin$test), type = "response")

  # 2. Proteins only: LASSO
  prot_fit  <- cv.glmnet(prot$train, y[train], family = "binomial", foldid = inner, standardize = FALSE)
  prot_risk <- predict(prot_fit, prot$test, s = "lambda.min", type = "response")[, 1]

  # 3. Clinical + proteins: LASSO, with age, sex, and BMI never penalized
  both_fit  <- cv.glmnet(cbind(clin$train, prot$train), y[train], family = "binomial", foldid = inner,
                         standardize = FALSE, penalty.factor = c(0, 0, 0, rep(1, ncol(prot$train))))
  both_risk <- predict(both_fit, cbind(clin$test, prot$test), s = "lambda.min", type = "response")[, 1]

  coefs <- coef(both_fit, s = "lambda.min")[, 1]
  list(risk   = tibble(row = which(test), clinical = clin_risk, proteins = prot_risk, both = both_risk),
       chosen = setdiff(names(coefs)[coefs != 0], c("(Intercept)", "age", "sex_male", "bmi")))
}

### All five folds, in parallel

`parallel::mclapply()` is `map()` on several processors at once: each fold runs on its own. ⏱️ 5 to 20 minutes on the default app. Start it, then discuss: **what would you expect proteins to add, beyond BMI?**

In [ ]:
start <- Sys.time()
folds <- parallel::mclapply(1:5, run_fold, mc.cores = min(5, CORES))
Sys.time() - start

failed <- map_lgl(folds, ~ inherits(.x, "try-error"))
if (any(failed)) stop("Fold ", which(failed)[1], " failed: ", folds[[which(failed)[1]]])

predictions <- map(folds, "risk") %>% list_rbind() %>% arrange(row) %>% mutate(y = y[row])
proteins_per_fold <- map_int(folds, ~ length(.x$chosen))
proteins_per_fold

**🔍 Read the output:** how long it took, then the number of proteins LASSO kept in each of the 5 folds, in the clinical + proteins model. If they differ a lot from fold to fold, the panel is **unstable**: we look at which proteins in Part 7.

If a fold failed with *cannot allocate memory*, shut down other kernels, or change `mc.cores = min(5, CORES)` to `mc.cores = 2` (slower, less memory).

## Part 6. Do proteins add anything?

### AUC, with a confidence interval

Every person now has three predicted risks, each from models that never saw them. `auc()` computes the AUC directly: it's the chance that a random case was given a higher risk than a random non-case, which is what **ranks** measure. (It gives the same answer as `roc_auc()` in 7.1, but is fast enough to run thousands of times.)

The **bootstrap** resamples the people with replacement, 1,000 times, and recomputes everything. The middle 95% of the results is the **95% confidence interval**. Because each resample uses the **same** people for all three models, the **difference** between models is measured precisely, even when each AUC on its own is uncertain.

In [ ]:
auc <- function(y, risk) {
  r  <- rank(risk)
  n1 <- sum(y == 1); n0 <- sum(y == 0)
  (sum(r[y == 1]) - n1 * (n1 + 1) / 2) / (n1 * n0)   # the Mann-Whitney formula for the AUC
}
brier <- function(y, risk) mean((y - risk)^2)

MODELS <- c(clinical = "Clinical: age, sex, BMI", proteins = "Proteins only (LASSO)",
            both = "Clinical + proteins (LASSO)")

set.seed(2026)
boot <- map(1:1000, function(b) {
  i <- sample(nrow(predictions), replace = TRUE)
  d <- predictions[i, ]
  tibble(clinical = auc(d$y, d$clinical), proteins = auc(d$y, d$proteins), both = auc(d$y, d$both))
}) %>% list_rbind()

tibble(model    = MODELS,
       auc      = map_dbl(names(MODELS), ~ auc(predictions$y, predictions[[.x]])),
       ci_low   = map_dbl(names(MODELS), ~ quantile(boot[[.x]], 0.025)),
       ci_high  = map_dbl(names(MODELS), ~ quantile(boot[[.x]], 0.975)),
       brier    = map_dbl(names(MODELS), ~ brier(predictions$y, predictions[[.x]]))) %>%
  bind_rows(tibble(model = "Everyone gets the average risk", auc = 0.5,
                   brier = brier(predictions$y, rep(mean(predictions$y), nrow(predictions))))) %>%
  mutate(across(c(auc, ci_low, ci_high), ~ round(.x, 3)), brier = round(brier, 4))

**🔍 Read the output:** one row per model: the cross-validated AUC, its 95% confidence interval, and the Brier score (lower is better; 7.2).

- Compare **Clinical** with **Clinical + proteins**: that's the added value.
- **Proteins only** shows how much the proteins know **on their own**. If it's close to the clinical model, the proteins may be largely **re-measuring** BMI and age: Part 7 checks.
- Don't compare these AUCs with 7.1's: these are **different people** (those with proteomics), with a different time zero.

In [ ]:
diff <- boot$both - boot$clinical
tibble(comparison  = "Clinical + proteins minus clinical",
       auc_gain    = round(auc(predictions$y, predictions$both) - auc(predictions$y, predictions$clinical), 3),
       ci_low      = round(quantile(diff, 0.025), 3),
       ci_high     = round(quantile(diff, 0.975), 3),
       pct_of_resamples_above_0 = round(100 * mean(diff > 0), 1))

**🔍 Read the output:** the gain in AUC from adding proteins, with its 95% confidence interval.

- **The interval is above 0:** the proteins add something, and the gain is unlikely to be chance.
- **The interval includes 0:** we can't tell, with this many cases, whether proteins add anything.
- Either way, ask whether the gain is **big enough to matter**. 7.2 found that boosting added 0.008 for free; is a protein panel worth its cost for a gain like this one?

### Calibration, in fifths

As in 7.1, but in **fifths** of predicted risk instead of tenths: with a few hundred cases, tenths would have too few cases each. Counts of 1 to 20 are hidden, and the next smallest too if the hidden ones add up to 1 to 20: their sum could be worked out from the total.

In [ ]:
# Hide counts of 1 to 20. The hidden counts add up to the total minus the shown ones, so if
# they add up to 1 to 20 (for example, only one is hidden), hide the next smallest too.
hide_with_total <- function(n) {
  hidden <- n <= 20
  while (any(hidden) && !all(hidden) && sum(n[hidden]) %in% 1:20) {
    hidden[which.min(replace(n, hidden, Inf))] <- TRUE
  }
  replace(n, hidden, NA)
}

calibration <- function(model) {
  tibble(risk = predictions[[model]], y = predictions$y) %>%
    mutate(risk_group = ntile(risk, 5)) %>%
    group_by(risk_group) %>%
    summarise(people = n(), predicted_pct = round(100 * mean(risk), 2), n = sum(y)) %>%
    mutate(n = hide_with_total(n),
           observed_pct = round(100 * n / people, 2), model = MODELS[[model]]) %>%
    rename(new_t2d = n)
}

bind_rows(calibration("clinical"), calibration("both")) %>%
  select(model, risk_group, people, predicted_pct, new_t2d, observed_pct)

**🔍 Read the output:** for each model, five groups from lowest to highest predicted risk. Predicted and observed percentages should be close. A better model **spreads** people further apart: a lower risk in the bottom fifth, a higher one in the top fifth.

## Part 7. The protein panel

### One final model, on everyone

Cross-validation told us **how well** the approach works. To see **which** proteins it uses, we fit the clinical + proteins model once more, on everyone. ⏱️ 1 to 5 minutes.

In [ ]:
all_p <- standardize(proteins, proteins)$train
all_c <- standardize(clinical, clinical)$train

set.seed(2026)
final_fit <- cv.glmnet(cbind(all_c, all_p), y, family = "binomial", foldid = assign_folds(y, 5),
                       standardize = FALSE, penalty.factor = c(0, 0, 0, rep(1, ncol(all_p))))

final_coefs <- coef(final_fit, s = "lambda.min")[, 1]
panel <- tibble(OlinkID = names(final_coefs), coefficient = final_coefs) %>%
  filter(coefficient != 0, !OlinkID %in% c("(Intercept)", "age", "sex_male", "bmi"))

nrow(panel)   # proteins in the final panel

**🔍 Read the output:** the number of proteins LASSO kept, out of about 5,000.

### Which proteins, and are they stable?

For each protein in the panel: its **odds ratio per SD** (from the model, with age, sex, BMI, and the other proteins in it), how many of the **5 outer folds** also chose it, and its **correlation with BMI**. The protein names come from the file's `Assay` column, kept in Part 3.

In [ ]:
times_chosen <- table(factor(unlist(map(folds, "chosen")), levels = colnames(proteins)))

panel_table <- panel %>%
  left_join(protein_names, by = "OlinkID") %>%
  mutate(odds_ratio_per_sd = round(exp(coefficient), 2),
         folds_chosen      = as.integer(times_chosen[OlinkID]),
         cor_with_bmi      = round(map_dbl(OlinkID, ~ cor(proteins[, .x], model_people$bmi, use = "complete.obs")), 2)) %>%
  arrange(desc(abs(coefficient))) %>%
  select(Assay, OlinkID, odds_ratio_per_sd, folds_chosen, cor_with_bmi)

head(panel_table, 20)

panel_table %>% count(folds_chosen, name = "proteins")   # how stable is the panel?

**🔍 Read the output:** the 20 proteins with the largest effects, then how many panel proteins were chosen in 0 to 5 of the outer folds. (These are counts of **proteins**, not people.)

- **Odds ratio per SD** above 1: a higher level goes with higher risk, at the same age, sex, BMI, and other proteins. Below 1: lower risk.
- **`folds_chosen` 5 of 5:** a **stable** protein, chosen whatever the data. **0 or 1 of 5:** chosen this time, but likely to change in another sample. LASSO picks **one** of a group of proteins that move together, sometimes this one, sometimes its neighbour.
- **`cor_with_bmi`** far from 0: the protein tracks body fat. It may add a **more precise** measure of what BMI only roughly captures.

**Not cause:** a protein in the panel goes **with** future diabetes. It may be a cause, a consequence of early disease, or a marker of something else.

## Your turn

1. **The one-SE rule.** In `run_fold()`, change the three `s = "lambda.min"` to `s = "lambda.1se"`, and rerun Parts 5 and 6. How many proteins are kept now, and what happens to the AUC gain?
2. **More clinical data.** Add `sbp` (systolic blood pressure, in 7.1's file) to the clinical model: in `select()` in Part 2, in `clinical` in Part 5, and one more `0` in `penalty.factor`. Does the protein gain shrink?
3. **A smaller panel.** Take the 10 proteins with the largest effects from `panel_table`, and fit an ordinary logistic regression with age, sex, BMI, and those 10, by cross-validation. Is a 10-protein panel almost as good? (Careful: these 10 were chosen using **everyone**, so this is optimistic. Why?)

## 💬 Discuss

1. A protein panel costs money and a blood draw; BMI costs a tape measure and scales. How big an AUC gain would justify the panel, and for whom?
2. If most panel proteins correlate with BMI, are they telling us anything new about diabetes?
3. We don't know how people were chosen for proteomics. How could that make the panel work worse in the rest of *All of Us*?

## ⚠️ Common errors

| Message or problem | Means | Fix |
|---|---|---|
| *ml_t2d_2yr.csv not found* (or similar) | 7.1 hasn't been run in this workspace | run 7.1 to the end |
| *genomics_metrics.tsv* not found | the genome QC folder moved, or has a different name | `list.files()` the `aux/qc` folder, as in the ML data check, section 6 |
| `stopifnot(!anyNA(col_nums))` fails | a column in the protein file was renamed | print `npx_columns` and update `wanted` |
| The app runs out of memory, or a fold fails with *cannot allocate memory* | five folds at once, on 5,000 proteins | shut down other kernels; or set `mc.cores = 2` |
| `Fold 1 failed: ...` | an error inside `run_fold()` | run `run_fold(1)` on its own to see the full message |
| All folds keep 0 proteins | the penalty is so strong that nothing gets in | normal with `lambda.1se` and few cases; with `lambda.min`, check the cases per fold |

## 🧑‍🏫 Instructor notes

- **The key idea is nested cross-validation.** Draw the two loops on the board. Ask: what goes wrong if we choose the penalty using all the data, and then cross-validate? (The test folds helped choose the penalty: optimistic.)
- **Start the long cells before discussing.** Part 3 (5 to 8 minutes) and Part 5 (5 to 20) are good moments for Part 1's questions.
- **Expect an unstable panel.** With a few hundred cases, many proteins are chosen in only one or two folds. That's the honest finding, and the reason published protein panels often fail to replicate.
- **Time zero matters more here than anywhere else in the series:** proteins change with diabetes, so a blood draw after diagnosis would make the panel look excellent, and be useless.
- **Don't publish two versions of these counts** (for example, with enrollment and with the blood draw as time zero): their differences would be small counts.

## Key takeaways

- With **more predictors than cases**, ordinary regression memorizes; **LASSO** keeps a small panel.
- When cases are few, **nested cross-validation** uses everyone for testing, with an inner loop for tuning.
- Ask what new data **adds** to cheap predictors, on the same people, with a **confidence interval** for the difference.
- A LASSO panel can be **unstable**: check how often each protein is chosen.
- **Time zero** must come before the outcome: for blood markers, it's the day the blood was drawn.

## Clean up

**Clear outputs** before committing or sharing: **Edit → Clear Outputs of All Cells**, then save. This notebook saves no files.